# Pattern 1: Single-Agent System

> **Google Cloud Pattern Reference**: [Single-Agent System](https://docs.cloud.google.com/architecture/choose-design-pattern-agentic-ai-system#single-agent-system)

A single agent uses an AI model, a defined set of tools, and a system prompt to autonomously handle a user request.

**When to use:**
- Tasks requiring multiple steps + external data
- Early-stage agent development
- Customer support, research assistants, data lookup

**Trade-offs:**
- ✅ Simple to build, debug, and maintain
- ✅ Lower cost (fewer model calls)
- ⚠️ Performance degrades with too many tools
- ⚠️ Not suitable for highly complex, multi-domain tasks

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

for var in ['OPENAI_API_KEY', 'OPENAI_BASE_URL', 'OPENAI_API_BASE']:
    if os.getenv(var):
        del os.environ[var]

os.environ["OPENAI_API_KEY"] = os.getenv("OPEN_ROUTER_KEY")
os.environ["OPENAI_BASE_URL"] = "https://openrouter.ai/api/v1"

### 1a. Basic Single Agent — No Tools
The simplest form: model + instructions.

In [2]:
from agno.agent import Agent
from agno.models.openai import OpenAIChat

customer_support_agent = Agent(
    name="Customer Support Agent",
    model=OpenAIChat(id="google/gemini-2.5-flash"),
    description="You are a helpful customer support agent for a tech company.",
    instructions=[
        "Always be polite and empathetic.",
        "If you cannot resolve an issue, escalate to a human agent.",
        "Provide clear step-by-step troubleshooting when needed.",
    ],
    markdown=True,
)

customer_support_agent.print_response(
    "My app keeps crashing when I open it. I'm on iOS 17.",
    stream=True,
)

Output()

### 1b. Single Agent with Tools
Add tools to give the agent access to external data. This is what separates agents from simple LLM calls.

In [3]:
from agno.agent import Agent
from agno.models.openai import OpenAIChat
from agno.tools.duckduckgo import DuckDuckGoTools
from agno.tools.yfinance import YFinanceTools

# Research agent — single agent with multiple tools
research_agent = Agent(
    name="Research Agent",
    model=OpenAIChat(id="google/gemini-2.5-flash"),
    tools=[
        DuckDuckGoTools(),
        YFinanceTools(enable_stock_price=True, enable_company_info=True),
    ],
    instructions=[
        "Search for current information before answering.",
        "Always cite your sources.",
        "Use tables to display financial data.",
    ],
    
    markdown=True,
)

research_agent.print_response(
    "What is the current stock price of NVIDIA and what are analysts saying about it?",
    stream=True,
)

Output()

### 1c. Single Agent with Reasoning (Extended Thinking)
Enable the model's extended reasoning for complex problems. This is Agno's built-in support for models like Gemini 2.5 Flash.

In [4]:
from agno.agent import Agent
from agno.models.openai import OpenAIChat

reasoning_agent = Agent(
    model=OpenAIChat(id="google/gemini-2.5-flash"),
    reasoning=True,
    markdown=True,
    structured_outputs=True,
)

task = (
    "Three missionaries and three cannibals need to cross a river. "
    "The boat holds 2 people. If cannibals ever outnumber missionaries on either bank, "
    "the missionaries will be eaten. Show me a step-by-step solution as an ASCII diagram."
)

reasoning_agent.print_response(task, stream=True, show_full_reasoning=True)

INFO Reasoning model: OpenAIChat is not a native reasoning model, defaulting to manual Chain-of-Thought reasoning

Output()

### 1d. Single Agent with Persistent Memory (Agno 2.x)

Agents can persist conversation history across `print_response` calls using `SqliteDb`.

**Agno 2.x API** (renamed from phidata/early Agno):
| Old param | New param |
|---|---|
| `storage=SqliteStorage(...)` | `db=SqliteDb(...)` |
| `add_history_to_messages=True` | `add_history_to_context=True` |
| `num_history_responses=N` | `num_history_runs=N` |

> **Important**: `add_history_to_context` requires `db=` to be set. Without it, Agno silently skips history injection.

In [5]:
from agno.agent import Agent
from agno.models.openai import OpenAIChat
from agno.db.sqlite import SqliteDb  # pip install sqlalchemy

persistent_agent = Agent(
    name="Persistent Research Agent",
    model=OpenAIChat(id="google/gemini-2.5-flash"),
    db=SqliteDb(db_file="agents.db"),  # persists to disk; use db_url="sqlite:///:memory:" for in-memory
    add_history_to_context=True,       # inject previous turns into each new request
    num_history_runs=3,                # how many prior turns to include
    markdown=True,
)

# First session
persistent_agent.print_response(
    "I am researching agentic AI patterns. Start with the single-agent pattern."
)



Output()

In [6]:
# Turn 2 — verify memory: agent should recall name and topic
result = persistent_agent.run("what was i researching?")
print(result.content)

You stated that you are researching **agentic AI patterns**.

I then started discussing the first of these patterns, the **single-agent pattern**.


In [7]:
import sqlite3

conn = sqlite3.connect("agents.db")
cursor = conn.cursor()

cursor.execute("SELECT agent_id, runs FROM agno_sessions ORDER BY created_at DESC LIMIT 1")
agent_id, runs_raw = cursor.fetchone()

print(f"Agent ID: {agent_id}")
print(f"Runs raw type: {type(runs_raw)}")
print(f"Runs raw value: {repr(runs_raw[:500])}")  # first 500 chars raw

conn.close()

Agent ID: persistent-research-agent
Runs raw type: <class 'str'>
Runs raw value: '"[{\\"run_id\\": \\"bb74c862-019b-4b89-a479-8d6d77d5ee0f\\", \\"agent_id\\": \\"persistent-research-agent\\", \\"agent_name\\": \\"Persistent Research Agent\\", \\"session_id\\": \\"7c796067-ea68-42de-9eb9-587bfa1b47ee\\", \\"content\\": \\"## Agentic AI Patterns: The Single-Agent Pattern\\\\n\\\\nThe **single-agent pattern** is the most fundamental and straightforward approach to designing agentic AI systems. It involves a single, autonomous AI agent that is responsible for achieving a specific goal or set of goals. '


### Summary

| Capability | Code |
|-----------|------|
| Basic agent | `Agent(model=..., instructions=[...])` |
| With tools | `Agent(tools=[DuckDuckGoTools(), ...])` |
| With reasoning | `Agent(reasoning=True)` |
| With memory | `Agent(db=SqliteDb(...), add_history_to_context=True)` |

👉 **Next**: `2_tools.ipynb` — deep dive into building custom tools for agents.